# Binary Shape Analysis
### OpenCV | Pattern Matching, Components, Contours & Blobs

This notebook continues **Binary_Morphology_Lab** and covers lessons 23–33. We use classic scikit-image sample images (Coins, Horse, Page and Text) and controlled synthetic masks to investigate classical object detection without a trained model.

**Learning objectives:** identify binary patterns, extract lines, label connected regions, measure shapes, localize words, count coins and detect blobs.

In [1]:
# Uncomment if necessary:
# %pip install opencv-python numpy matplotlib scikit-image
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import data

print('OpenCV:', cv2.__version__)

def show(images, titles, figsize=(15, 5)):
    fig, axes = plt.subplots(1, len(images), figsize=figsize, squeeze=False)
    for ax, image, title in zip(axes[0], images, titles):
        ax.imshow(image, cmap='gray', vmin=0, vmax=255) if image.ndim == 2 else ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

coins = data.coins()
page = data.page()
text = data.text()
horse = np.where(data.horse(), 0, 255).astype(np.uint8)
show([coins, page, text, horse], ['Coins', 'Page', 'Text', 'Horse'])

OpenCV: 4.13.0


/opt/pyvenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/tmp/ipykernel_608/1646971176.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 1. Hit-or-Miss Transform — Lesson 23
Hit-or-miss locates a specific binary neighborhood rather than simply expanding or shrinking it. A ternary kernel contains `1` for required foreground, `-1` for required background and `0` for ignored positions.

$$A\circledast B=(A\ominus B_1)\cap(A^c\ominus B_2)$$

We detect the endpoints of thin horizontal and vertical lines using four rotated kernels. The center must be foreground and exactly one cardinal neighbor must be foreground.

In [2]:
pattern = np.zeros((100, 160), np.uint8)
cv2.line(pattern, (20, 25), (65, 25), 255, 1)
cv2.line(pattern, (100, 20), (100, 75), 255, 1)
cv2.line(pattern, (25, 70), (65, 70), 255, 1)
endpoint_kernel = np.array([[-1,-1,-1],[-1,1,1],[-1,-1,-1]], dtype=np.int8)
hits = np.zeros_like(pattern)
for k in range(4):
    rotated = np.ascontiguousarray(np.rot90(endpoint_kernel, k))
    response = cv2.morphologyEx(pattern, cv2.MORPH_HITMISS, rotated)
    hits = cv2.bitwise_or(hits, response)
marked = cv2.cvtColor(pattern, cv2.COLOR_GRAY2BGR)
for y, x in np.argwhere(hits > 0):
    cv2.circle(marked, (int(x), int(y)), 4, (0, 0, 255), 1)
print('Endpoints:', np.count_nonzero(hits))
show([pattern, hits, marked], ['Input', 'Hit-or-miss response', 'Endpoints'])

Endpoints: 6


/tmp/ipykernel_608/1646971176.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Horizontal and Vertical Line Removal — Lesson 24
Long rectangular kernels select structures aligned with their orientation. Opening with `(length, 1)` extracts horizontal lines; `(1, length)` extracts vertical lines. Subtracting their masks removes the extracted lines. OpenCV specifies kernel dimensions as **width, height**.

We generate a ruled form with text so that the behavior is easy to verify. Intersections with characters may also be removed.

In [3]:
form = np.full((220, 460), 255, np.uint8)
for y in [20, 80, 140, 200]:
    cv2.line(form, (10,y), (450,y), 0, 2)
for x in [10, 230, 450]:
    cv2.line(form, (x,20), (x,200), 0, 2)
for i, words in enumerate([('ITEM','COUNT'),('Apple','12'),('Pear','25')]):
    for x, word in zip([25,245], words):
        cv2.putText(form, word, (x,60+i*60), cv2.FONT_HERSHEY_SIMPLEX, .8, 0, 2)
mask = cv2.bitwise_not(form)
horizontal = cv2.morphologyEx(mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT,(45,1)))
vertical = cv2.morphologyEx(mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT,(1,45)))
lines = cv2.bitwise_or(horizontal, vertical)
clean_form = cv2.bitwise_not(cv2.subtract(mask, lines))
show([form, horizontal, vertical, clean_form], ['Form','Horizontal lines','Vertical lines','Cleaned form'])

/tmp/ipykernel_608/1646971176.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Connected Components — Lesson 25
Connected component labeling assigns an integer ID to every connected foreground region. Four-connectivity uses cardinal neighbors; eight-connectivity also includes diagonals. Label zero is background.

`connectedComponentsWithStats` returns labels, bounding boxes, pixel areas and centroids.

In [4]:
diagonal = np.zeros((9,9), np.uint8)
diagonal[2,2] = diagonal[3,3] = diagonal[4,4] = 255
for connectivity in [4,8]:
    count, labels = cv2.connectedComponents(diagonal, connectivity=connectivity)
    print(f'{connectivity}-connectivity: {count-1} objects')
    plt.figure(figsize=(3,3)); plt.imshow(labels, cmap='tab10'); plt.title(f'{connectivity}-connectivity'); plt.show()

4-connectivity: 3 objects
8-connectivity: 1 objects


/tmp/ipykernel_608/511812510.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.figure(figsize=(3,3)); plt.imshow(labels, cmap='tab10'); plt.title(f'{connectivity}-connectivity'); plt.show()


In [5]:
shapes = np.zeros((220,360), np.uint8)
cv2.circle(shapes,(65,65),35,255,-1)
cv2.rectangle(shapes,(140,35),(205,100),255,-1)
cv2.fillPoly(shapes,[np.array([[255,110],[325,110],[290,25]])],255)
cv2.ellipse(shapes,(130,165),(48,22),20,0,360,255,-1)
cv2.circle(shapes,(340,190),2,255,-1)
count, labels, stats, centers = cv2.connectedComponentsWithStats(shapes, connectivity=8)
colored = np.zeros((*shapes.shape,3),np.uint8)
rng = np.random.default_rng(42)
for i in range(1,count):
    colored[labels==i] = rng.integers(70,256,3,dtype=np.uint8)
    x,y,w,h,area = stats[i]
    if area < 50: continue
    cx,cy = centers[i]
    cv2.rectangle(colored,(x,y),(x+w-1,y+h-1),(255,255,255),1)
    cv2.circle(colored,(round(cx),round(cy)),3,(0,0,255),-1)
    print(f'ID {i}: area={area}, centroid=({cx:.1f}, {cy:.1f})')
show([shapes,colored],['Binary objects','Labeled components'])

ID 1: area=3096, centroid=(290.0, 81.6)
ID 2: area=3853, centroid=(65.0, 65.0)
ID 3: area=4356, centroid=(172.5, 67.5)
ID 4: area=3425, centroid=(130.0, 165.0)


/tmp/ipykernel_608/1646971176.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Finding Contours — Lesson 26
Contours are ordered boundary coordinates. Unlike connected components, they describe boundaries rather than assigning IDs to all foreground pixels.

`RETR_EXTERNAL` returns outer boundaries; `RETR_LIST` returns all without nesting; `RETR_CCOMP` provides two levels; `RETR_TREE` preserves full nesting. `CHAIN_APPROX_SIMPLE` compresses straight segments, while `CHAIN_APPROX_NONE` retains all boundary steps.

In [6]:
nested = np.zeros((200,300),np.uint8)
cv2.rectangle(nested,(25,20),(275,180),255,-1)
cv2.circle(nested,(150,100),58,0,-1)
cv2.circle(nested,(150,100),24,255,-1)
outputs=[]; titles=[]
for mode,name in [(cv2.RETR_EXTERNAL,'External'),(cv2.RETR_LIST,'List'),(cv2.RETR_CCOMP,'CCOMP'),(cv2.RETR_TREE,'Tree')]:
    contours,hierarchy = cv2.findContours(nested,mode,cv2.CHAIN_APPROX_SIMPLE)
    canvas=cv2.cvtColor(nested,cv2.COLOR_GRAY2BGR)
    cv2.drawContours(canvas,contours,-1,(0,255,0),2)
    outputs.append(canvas); titles.append(f'{name}: {len(contours)}')
    print(name, hierarchy[0].tolist())
show(outputs,titles)

External [[-1, -1, -1, -1]]
List [[1, -1, -1, -1], [2, 0, -1, -1], [-1, 1, -1, -1]]
CCOMP [[1, -1, -1, -1], [-1, 0, 2, -1], [-1, -1, -1, 1]]
Tree [[-1, -1, 1, -1], [-1, -1, 2, 0], [-1, -1, -1, 1]]


/tmp/ipykernel_608/1646971176.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
for method in [cv2.CHAIN_APPROX_NONE,cv2.CHAIN_APPROX_SIMPLE]:
    cs,_=cv2.findContours(nested,cv2.RETR_EXTERNAL,method)
    print('Method',method,'stored points:',len(cs[0]))

Method 1 stored points: 820
Method 2 stored points: 4


## 5. Convex Hull — Lesson 27
A convex hull is the smallest convex set enclosing the contour points. Convexity defects describe inward deviations from the hull. We use the classic Horse silhouette and a synthetic star for a clearer defect demonstration.

For `convexityDefects`, request hull indices using `returnPoints=False`. The returned depth is fixed-point and must be divided by 256.

In [8]:
cs,_=cv2.findContours(horse,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
horse_contour=max(cs,key=cv2.contourArea)
hull=cv2.convexHull(horse_contour)
canvas=cv2.cvtColor(horse,cv2.COLOR_GRAY2BGR)
cv2.drawContours(canvas,[horse_contour],-1,(0,255,0),2)
cv2.drawContours(canvas,[hull],-1,(0,0,255),2)
show([horse,canvas],['Horse silhouette','Contour (green) and hull (red)'])
angles=np.arange(10)*np.pi/5-np.pi/2
radii=np.tile([80,35],5)
star_points=np.column_stack([110+radii*np.cos(angles),110+radii*np.sin(angles)]).astype(np.int32)
star=np.zeros((220,220),np.uint8)
cv2.fillPoly(star,[star_points],255)
cs,_=cv2.findContours(star,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
star_c=cs[0]
hull_idx=cv2.convexHull(star_c,returnPoints=False)
defects=cv2.convexityDefects(star_c,hull_idx)
canvas=cv2.cvtColor(star,cv2.COLOR_GRAY2BGR)
cv2.drawContours(canvas,[cv2.convexHull(star_c)],-1,(0,255,0),2)
if defects is not None:
    for s,e,f,d in defects[:,0]:
        if d/256 > 5:
            cv2.circle(canvas,tuple(star_c[f,0]),4,(0,0,255),-1)
            print(f'Defect depth: {d/256:.2f} px')
show([star,canvas],['Star','Hull and significant defects'])

Defect depth: 29.00 px
Defect depth: 28.71 px
Defect depth: 30.04 px
Defect depth: 29.00 px
Defect depth: 30.04 px


/tmp/ipykernel_608/1646971176.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Bounding Shapes — Lesson 28
`boundingRect` returns an axis-aligned rectangle. `minAreaRect` returns a minimum-area rotated rectangle. `minEnclosingCircle` returns the smallest enclosing circle. These summarize object location and size, but do not necessarily match its true geometry.

In [9]:
cs,_=cv2.findContours(shapes,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
contour=max([c for c in cs if cv2.boundingRect(c)[1]>110],key=cv2.contourArea)
views=[cv2.cvtColor(shapes,cv2.COLOR_GRAY2BGR) for _ in range(3)]
x,y,w,h=cv2.boundingRect(contour)
cv2.rectangle(views[0],(x,y),(x+w-1,y+h-1),(0,255,0),2)
rect=cv2.minAreaRect(contour)
box=np.rint(cv2.boxPoints(rect)).astype(np.int32)
cv2.drawContours(views[1],[box],0,(0,255,0),2)
(cx,cy),radius=cv2.minEnclosingCircle(contour)
cv2.circle(views[2],(round(cx),round(cy)),int(np.ceil(radius)),(0,255,0),2)
show(views,['Axis-aligned','Rotated rectangle','Enclosing circle'])

/tmp/ipykernel_608/1646971176.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Moments and Geometric Features — Lesson 29
Image moments describe spatial distributions:

$$m_{pq}=\sum_x\sum_y x^p y^q I(x,y),\qquad C_x=m_{10}/m_{00},\quad C_y=m_{01}/m_{00}$$

For contour moments, OpenCV integrates over the enclosed polygon. Contour area is not necessarily the same as the number of foreground pixels.

$$\text{Circularity}=4\pi A/P^2,\qquad \text{Solidity}=A/A_{hull},\qquad \text{Extent}=A/(wh)$$

Aspect ratio is $w/h$. These features can be used for rule-based classification or as inputs to a machine-learning model.

In [10]:
def shape_features(c):
    area=cv2.contourArea(c)
    perimeter=cv2.arcLength(c,True)
    hull_area=cv2.contourArea(cv2.convexHull(c))
    x,y,w,h=cv2.boundingRect(c)
    M=cv2.moments(c)
    center=(M['m10']/M['m00'],M['m01']/M['m00']) if M['m00'] else None
    return dict(area=area, perimeter=perimeter, circularity=4*np.pi*area/perimeter**2 if perimeter else 0,
                solidity=area/hull_area if hull_area else 0, extent=area/(w*h), aspect_ratio=w/h, center=center)
canvas=cv2.cvtColor(shapes,cv2.COLOR_GRAY2BGR)
for c in cs:
    f=shape_features(c)
    if f['area']<50: continue
    x,y=map(round,f['center'])
    cv2.circle(canvas,(x,y),3,(0,0,255),-1)
    print({k:round(v,3) if isinstance(v,(float,np.floating)) else v for k,v in f.items()})
show([canvas],['Centroids'])

{'area': 3321.0, 'perimeter': 239.966, 'circularity': 0.725, 'solidity': 0.986, 'extent': 0.674, 'aspect_ratio': 1.755, 'center': (129.9953327311051, 165.0034628124059)}
{'area': 4225.0, 'perimeter': 260.0, 'circularity': 0.785, 'solidity': 1.0, 'extent': 0.97, 'aspect_ratio': 1.0, 'center': (172.5, 67.5)}
{'area': 3754.0, 'perimeter': 230.794, 'circularity': 0.886, 'solidity': 0.984, 'extent': 0.745, 'aspect_ratio': 1.0, 'center': (65.0, 65.0)}
{'area': 2975.0, 'perimeter': 268.995, 'circularity': 0.517, 'solidity': 0.975, 'extent': 0.487, 'aspect_ratio': 0.826, 'center': (290.0, 81.68683473389355)}


/tmp/ipykernel_608/1646971176.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Polygon Approximation — Lesson 30
Douglas–Peucker simplifies a contour by recursively retaining points whose deviation exceeds a tolerance. `approxPolyDP` accepts an epsilon in pixels; a useful starting point is a fraction of the perimeter. Increasing epsilon usually reduces the number of vertices, at the cost of geometric detail.

In [11]:
views=[]; titles=[]
for fraction in [.005,.02,.06,.12]:
    approx=cv2.approxPolyDP(star_c,fraction*cv2.arcLength(star_c,True),True)
    canvas=cv2.cvtColor(star,cv2.COLOR_GRAY2BGR)
    cv2.drawContours(canvas,[approx],-1,(0,255,0),2)
    for point in approx[:,0]: cv2.circle(canvas,tuple(point),3,(0,0,255),-1)
    views.append(canvas); titles.append(f'{fraction:.3f}: {len(approx)} vertices')
show(views,titles)

/tmp/ipykernel_608/1646971176.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Experiment — Simple shape classification
Vertex count distinguishes triangles and quadrilaterals in a controlled scene. Circularity helps distinguish rounded shapes. A bounding-box ratio is only reliable for approximately axis-aligned squares; rotated or irregular shapes require more geometric checks.

In [12]:
canvas=cv2.cvtColor(shapes,cv2.COLOR_GRAY2BGR)
for c in cs:
    f=shape_features(c)
    if f['area']<50: continue
    a=cv2.approxPolyDP(c,.02*f['perimeter'],True)
    label='Triangle' if len(a)==3 else 'Square-like' if len(a)==4 and .9<f['aspect_ratio']<1.1 else 'Rectangle-like' if len(a)==4 else 'Round' if f['circularity']>.8 else 'Other'
    x,y,w,h=cv2.boundingRect(c)
    cv2.putText(canvas,label,(x,max(12,y-5)),cv2.FONT_HERSHEY_SIMPLEX,.4,(0,255,0),1)
show([canvas],['Rule-based classification'])

/tmp/ipykernel_608/1646971176.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Word Localization — Lesson 31
This is a localization task, not OCR. Adaptive thresholding produces white text strokes, then a horizontal dilation connects nearby characters. External contours provide candidate word boxes. Kernel width must bridge intra-word gaps without merging separate words.

In [13]:
word_mask=cv2.adaptiveThreshold(page,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,25,12)
joined=cv2.dilate(word_mask,cv2.getStructuringElement(cv2.MORPH_RECT,(9,1)))
word_cs,_=cv2.findContours(joined,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
canvas=cv2.cvtColor(page,cv2.COLOR_GRAY2BGR)
word_count=0
for c in word_cs:
    x,y,w,h=cv2.boundingRect(c)
    if w>=8 and 5<=h<=35 and w*h>=60:
        cv2.rectangle(canvas,(x,y),(x+w-1,y+h-1),(0,255,0),1)
        word_count+=1
print('Candidate regions (not verified word count):',word_count)
show([page,word_mask,joined,canvas],['Page','Adaptive mask','Dilation','Candidate words'],(17,5))

Candidate regions (not verified word count): 24


/tmp/ipykernel_608/1646971176.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Coin Counting — Lesson 32
The classic Coins image contains varying background illumination. A local background estimate (large morphological opening) helps isolate brighter coins. Thresholding, mask cleanup, contours and shape filters produce candidate detections. Touching or low-contrast coins can still be missed; this is a baseline, not a guaranteed exact counter.

In [14]:
smooth=cv2.GaussianBlur(coins,(5,5),0)
background=cv2.morphologyEx(smooth,cv2.MORPH_OPEN,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(65,65)))
corrected=cv2.subtract(smooth,background)
_,coin_mask=cv2.threshold(corrected,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
coin_mask=cv2.morphologyEx(coin_mask,cv2.MORPH_CLOSE,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(9,9)))
coin_mask=cv2.morphologyEx(coin_mask,cv2.MORPH_OPEN,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5)))
coin_cs,_=cv2.findContours(coin_mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
canvas=cv2.cvtColor(coins,cv2.COLOR_GRAY2BGR)
count=0
for c in sorted(coin_cs,key=lambda c:cv2.boundingRect(c)[0]):
    f=shape_features(c)
    if not (250<f['area']<3500 and f['circularity']>.45 and f['center']): continue
    count+=1
    cv2.drawContours(canvas,[c],-1,(0,255,0),2)
    cv2.putText(canvas,str(count),tuple(map(round,f['center'])),cv2.FONT_HERSHEY_SIMPLEX,.5,(0,0,255),1)
print('Detected coin candidates:',count)
show([coins,corrected,coin_mask,canvas],['Coins','Background corrected','Mask','Detections'])

Detected coin candidates: 22


/tmp/ipykernel_608/1646971176.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Blob Detection — Lesson 33
A blob is a locally distinct image region. `SimpleBlobDetector` thresholds the grayscale image at multiple levels, finds candidate regions, groups nearby centers across thresholds and filters them by color, area, circularity, convexity and inertia ratio.

The inertia ratio is the ratio of the smaller to larger eigenvalue of the region's second-moment matrix. A circular region approaches 1; an elongated region approaches 0. `KeyPoint.size` is an estimated diameter, not area.

In [15]:
spots=np.full((230,370),235,np.uint8)
for center,radius in [((55,55),18),((125,55),27),((210,65),12),((310,70),34),((65,165),23)]:
    cv2.circle(spots,center,radius,35,-1)
cv2.ellipse(spots,(190,165),(48,8),0,0,360,35,-1)
cv2.rectangle(spots,(285,140),(340,190),35,-1)

def make_detector(circularity=.75,inertia=.5):
    params=cv2.SimpleBlobDetector_Params()
    params.minThreshold=10; params.maxThreshold=230; params.thresholdStep=10
    params.filterByColor=True; params.blobColor=0
    params.filterByArea=True; params.minArea=100; params.maxArea=6000
    params.filterByCircularity=True; params.minCircularity=circularity
    params.filterByConvexity=True; params.minConvexity=.85
    params.filterByInertia=True; params.minInertiaRatio=inertia
    return cv2.SimpleBlobDetector_create(params)

views=[]; titles=[]
for circ,inertia in [(.5,.1),(.85,.6)]:
    keypoints=make_detector(circ,inertia).detect(spots)
    rendered=cv2.drawKeypoints(spots,keypoints,None,(0,0,255),cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    views.append(rendered); titles.append(f'Circularity {circ}: {len(keypoints)} blobs')
show([spots,*views],['Input',*titles])
for kp in keypoints: print('Center:',tuple(round(v,1) for v in kp.pt),'Diameter:',round(kp.size,1))

Center: (65.0, 165.0) Diameter: 46.7
Center: (210.0, 65.0) Diameter: 24.4
Center: (55.0, 55.0) Diameter: 36.8
Center: (310.0, 70.0) Diameter: 68.9
Center: (125.0, 55.0) Diameter: 54.9


/tmp/ipykernel_608/1646971176.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

| Lesson | Topic | Main OpenCV API |
|---|---|---|
| 23 | Pattern detection | `MORPH_HITMISS` |
| 24 | Line removal | `morphologyEx`, `subtract` |
| 25 | Connected components | `connectedComponentsWithStats` |
| 26 | Contours | `findContours` |
| 27 | Convex hull | `convexHull`, `convexityDefects` |
| 28 | Bounding shapes | `boundingRect`, `minAreaRect`, `minEnclosingCircle` |
| 29 | Moments and features | `moments`, `contourArea`, `arcLength` |
| 30 | Polygon approximation | `approxPolyDP` |
| 31 | Word localization | Thresholding + dilation + contours |
| 32 | Coin counting | Morphology + contour filtering |
| 33 | Blob detection | `SimpleBlobDetector_create` |

## Exercises
1. Modify the hit-or-miss kernel to detect a T-junction.
2. Compare component labels with contour boundaries on the same mask.
3. Investigate solidity for the star and horse silhouettes.
4. Tune word dilation width and manually inspect merged or split words.
5. Improve coin counting and compare detections with a manual count.
6. Vary blob circularity and inertia filters independently.

## References
- [Course repository](https://github.com/Alireza-Akhavan/class.vision)
- [OpenCV documentation](https://docs.opencv.org/4.x/)
- [scikit-image sample data and image attribution](https://scikit-image.org/docs/stable/api/skimage.data.html)

All code examples are independently written for this notebook. Sample images are provided by scikit-image; synthetic masks are generated locally.